# Library and Install

In [ ]:
!pip install lime
!pip install scikit-image

In [ ]:
# =========================
# Standard Libraries
# =========================
import os
import zipfile
import datetime

# =========================
# Numerical & Data Handling
# =========================
import numpy as np
import pandas as pd

# =========================
# Visualization
# =========================
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MultipleLocator

# =========================
# TensorFlow / Keras
# =========================
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model

# =========================
# Scikit-learn
# =========================
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# =========================
# Explainability (LIME)
# =========================
from lime import lime_image
from skimage.segmentation import mark_boundaries

# =========================
# Google Colab
# =========================
from google.colab import drive

# Manage Files

In [ ]:
drive.mount('/content/drive')

In [ ]:
zip_path = "/content/drive/MyDrive/Mango_Leaf_Disease.zip"

extract_dir = "/content/Mango_Leaf_Disease"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Dataset extracted to:", extract_dir)



```
Dataset extracted to: /content/Mango_Leaf_Disease
```



In [ ]:
print(os.listdir(extract_dir))



```
['Anthracnose', 'Gall Midge', 'Cutting Weevil', 'Sooty Mould', 'Bacterial Canker', 'Powdery Mildew', 'Die Back', 'Healthy']

```



# Data Processing

In [ ]:
extract_dir = '/content/Mango_Leaf_Disease'
classes = sorted(os.listdir(extract_dir))

filepaths = []
labels = []

for idx, cls in enumerate(classes):
    cls_folder = os.path.join(extract_dir, cls)
    for fname in os.listdir(cls_folder):
        if fname.endswith(('.jpg', '.png', '.jpeg')):
            filepaths.append(os.path.join(cls_folder, fname))
            labels.append(idx)

filepaths = np.array(filepaths)
labels = np.array(labels)

In [ ]:
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    filepaths,
    labels,
    test_size=800,
    stratify=labels,
    random_state=42
)

In [ ]:
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42
)

In [ ]:

train_df = pd.DataFrame({'filename': train_paths, 'class': train_labels})
val_df = pd.DataFrame({'filename': val_paths, 'class': val_labels})
test_df = pd.DataFrame({'filename': test_paths, 'class': test_labels})

In [ ]:
IMG_SIZE = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
class_names = sorted(os.listdir(extract_dir))
train_df['class'] = [class_names[i] for i in train_labels]
val_df['class']   = [class_names[i] for i in val_labels]
test_df['class']  = [class_names[i] for i in test_labels]

In [ ]:
train_data = train_datagen.flow_from_dataframe(
    train_df,
    x_col='filename',
    y_col='class',
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

val_data = val_test_datagen.flow_from_dataframe(
    val_df,
    x_col='filename',
    y_col='class',
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

test_data = val_test_datagen.flow_from_dataframe(
    test_df,
    x_col='filename',
    y_col='class',
    target_size=IMG_SIZE,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)



```
Found 3200 validated image filenames belonging to 8 classes.
Found 400 validated image filenames belonging to 8 classes.
Found 400 validated image filenames belonging to 8 classes.

```



In [ ]:
class_indices = train_data.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}

num_classes = len(class_indices)
seen_classes = set()
images = []
labels = []

for batch_imgs, batch_labels in train_data:
    for img, label_vec in zip(batch_imgs, batch_labels):
        class_idx = np.argmax(label_vec)
        if class_idx not in seen_classes:
            images.append(img)
            labels.append(idx_to_class[class_idx])
            seen_classes.add(class_idx)
        if len(seen_classes) == num_classes:
            break
    if len(seen_classes) == num_classes:
        break


plt.figure(figsize=(16, 4))
for i, (img, label) in enumerate(zip(images, labels)):
    plt.subplot(1, num_classes, i + 1)


    if img.max() <= 1.0:
        plt.imshow(img)
    else:
        plt.imshow(img.astype(np.uint8))

    plt.title(label)
    plt.axis('off')

plt.suptitle("One Image from Each Class", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
labels = train_data.classes

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))
print("Class weights:", class_weights)



```
Class weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0), 5: np.float64(1.0), 6: np.float64(1.0), 7: np.float64(1.0)}

```



# Model & Training

In [ ]:
base_model = ResNet50(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

In [ ]:
base_model.summary()

In [ ]:
active = [
    "conv1",
    "conv2_block1_1", "conv2_block1_2",
    "conv2_block2_1", "conv2_block2_2",
    "conv2_block3_1", "conv2_block3_2",
    "conv3_block1_1", "conv3_block1_2",
    "conv3_block2_1", "conv3_block2_2",
    "conv3_block3_1", "conv3_block3_2",
    "conv3_block4_1", "conv3_block4_2"
]

In [ ]:
for layer in base_model.layers:
    if any(name in layer.name for name in active):
        layer.trainable = True
        print("Unfrozen layer:", layer.name)



```
Unfrozen layer: conv1_pad
Unfrozen layer: conv1_conv
Unfrozen layer: conv1_bn
Unfrozen layer: conv1_relu
Unfrozen layer: conv2_block1_1_conv
Unfrozen layer: conv2_block1_1_bn
Unfrozen layer: conv2_block1_1_relu
Unfrozen layer: conv2_block1_2_conv
Unfrozen layer: conv2_block1_2_bn
Unfrozen layer: conv2_block1_2_relu
Unfrozen layer: conv2_block2_1_conv
Unfrozen layer: conv2_block2_1_bn
Unfrozen layer: conv2_block2_1_relu
Unfrozen layer: conv2_block2_2_conv
Unfrozen layer: conv2_block2_2_bn
Unfrozen layer: conv2_block2_2_relu
Unfrozen layer: conv2_block3_1_conv
Unfrozen layer: conv2_block3_1_bn
Unfrozen layer: conv2_block3_1_relu
Unfrozen layer: conv2_block3_2_conv
Unfrozen layer: conv2_block3_2_bn
Unfrozen layer: conv2_block3_2_relu
Unfrozen layer: conv3_block1_1_conv
Unfrozen layer: conv3_block1_1_bn
Unfrozen layer: conv3_block1_1_relu
Unfrozen layer: conv3_block1_2_conv
Unfrozen layer: conv3_block1_2_bn
Unfrozen layer: conv3_block1_2_relu
Unfrozen layer: conv3_block2_1_conv
Unfrozen layer: conv3_block2_1_bn
Unfrozen layer: conv3_block2_1_relu
Unfrozen layer: conv3_block2_2_conv
Unfrozen layer: conv3_block2_2_bn
Unfrozen layer: conv3_block2_2_relu
Unfrozen layer: conv3_block3_1_conv
Unfrozen layer: conv3_block3_1_bn
Unfrozen layer: conv3_block3_1_relu
Unfrozen layer: conv3_block3_2_conv
Unfrozen layer: conv3_block3_2_bn
Unfrozen layer: conv3_block3_2_relu
Unfrozen layer: conv3_block4_1_conv
Unfrozen layer: conv3_block4_1_bn
Unfrozen layer: conv3_block4_1_relu
Unfrozen layer: conv3_block4_2_conv
Unfrozen layer: conv3_block4_2_bn
Unfrozen layer: conv3_block4_2_relu
```



In [ ]:
for i, layer in enumerate(base_model.layers):
    print(i, layer.name, layer.trainable)




```
MODEL BUILDING RECORD
```


```
accuracy: 0.9994 - loss: 0.0101 - val_accuracy: 0.9762 - val_loss: 0.0845
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(8, activation='softmax')
])
```


```
accuracy: 0.9922 - loss: 0.0311 - val_accuracy: 0.9750 - val_loss: 0.0673
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(8, activation='softmax')
])
```



```
accuracy: 0.9931 - loss: 0.0310 - val_accuracy: 0.9775 - val_loss: 0.0674
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(8, activation='softmax')
])

```




```
accuracy: 0.9912 - loss: 0.0329 - val_accuracy: 0.9762 - val_loss: 0.0729
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(8, activation='softmax')
])
```



```
accuracy: 0.9994 - loss: 0.0088 - val_accuracy: 0.9787 - val_loss: 0.0766

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(8, activation='softmax')
])
```







```
Final Model
accuracy: 1.0000 - loss: 0.0020 - val_accuracy: 0.9975 - val_loss: 0.0060
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(2048, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(8, activation='softmax')
])
```



In [ ]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(2048, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(8, activation='softmax')
])

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()



```
Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2048)           │     4,196,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │        16,392 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 27,800,456 (106.05 MB)
 Trainable params: 5,193,224 (19.81 MB)
 Non-trainable params: 22,607,232 (86.24 MB)
```



In [ ]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=50,
    callbacks=[tensorboard_callback, early_stop]
)



```
Epoch 1/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 95s 641ms/step - accuracy: 0.5813 - loss: 1.2638 - val_accuracy: 0.1000 - val_loss: 2.1891
Epoch 2/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 553ms/step - accuracy: 0.8897 - loss: 0.4340 - val_accuracy: 0.1375 - val_loss: 2.2993
Epoch 3/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 82s 551ms/step - accuracy: 0.9506 - loss: 0.2246 - val_accuracy: 0.0950 - val_loss: 2.3318
Epoch 4/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 82s 553ms/step - accuracy: 0.9694 - loss: 0.1357 - val_accuracy: 0.1025 - val_loss: 2.2846
Epoch 5/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 56s 556ms/step - accuracy: 0.9825 - loss: 0.0900 - val_accuracy: 0.1625 - val_loss: 2.1161
Epoch 6/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 542ms/step - accuracy: 0.9878 - loss: 0.0660 - val_accuracy: 0.2550 - val_loss: 1.8544
Epoch 7/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 56s 553ms/step - accuracy: 0.9909 - loss: 0.0523 - val_accuracy: 0.4275 - val_loss: 1.4966
Epoch 8/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 56s 556ms/step - accuracy: 0.9919 - loss: 0.0417 - val_accuracy: 0.6250 - val_loss: 1.0116
Epoch 9/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 56s 558ms/step - accuracy: 0.9962 - loss: 0.0316 - val_accuracy: 0.8000 - val_loss: 0.5824
Epoch 10/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 551ms/step - accuracy: 0.9937 - loss: 0.0308 - val_accuracy: 0.9200 - val_loss: 0.2460
Epoch 11/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 56s 555ms/step - accuracy: 0.9962 - loss: 0.0238 - val_accuracy: 0.9775 - val_loss: 0.0890
Epoch 12/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 56s 560ms/step - accuracy: 0.9972 - loss: 0.0188 - val_accuracy: 0.9950 - val_loss: 0.0311
Epoch 13/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 57s 568ms/step - accuracy: 0.9984 - loss: 0.0172 - val_accuracy: 0.9950 - val_loss: 0.0227
Epoch 14/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 56s 559ms/step - accuracy: 0.9981 - loss: 0.0153 - val_accuracy: 0.9950 - val_loss: 0.0190
Epoch 15/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 56s 555ms/step - accuracy: 0.9987 - loss: 0.0127 - val_accuracy: 0.9975 - val_loss: 0.0165
Epoch 16/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 547ms/step - accuracy: 0.9981 - loss: 0.0115 - val_accuracy: 0.9925 - val_loss: 0.0216
Epoch 17/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 549ms/step - accuracy: 0.9991 - loss: 0.0097 - val_accuracy: 0.9950 - val_loss: 0.0149
Epoch 18/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 56s 555ms/step - accuracy: 0.9987 - loss: 0.0090 - val_accuracy: 0.9950 - val_loss: 0.0135
Epoch 19/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 550ms/step - accuracy: 0.9981 - loss: 0.0088 - val_accuracy: 0.9975 - val_loss: 0.0117
Epoch 20/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 552ms/step - accuracy: 0.9991 - loss: 0.0084 - val_accuracy: 0.9950 - val_loss: 0.0118
Epoch 21/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 551ms/step - accuracy: 0.9987 - loss: 0.0075 - val_accuracy: 0.9950 - val_loss: 0.0097
Epoch 22/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 548ms/step - accuracy: 0.9981 - loss: 0.0073 - val_accuracy: 0.9950 - val_loss: 0.0128
Epoch 23/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 54s 540ms/step - accuracy: 0.9987 - loss: 0.0063 - val_accuracy: 0.9950 - val_loss: 0.0119
Epoch 24/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 54s 541ms/step - accuracy: 0.9994 - loss: 0.0054 - val_accuracy: 0.9950 - val_loss: 0.0093
Epoch 25/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 547ms/step - accuracy: 0.9991 - loss: 0.0071 - val_accuracy: 0.9950 - val_loss: 0.0083
Epoch 26/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 54s 543ms/step - accuracy: 0.9994 - loss: 0.0047 - val_accuracy: 0.9975 - val_loss: 0.0071
Epoch 27/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 549ms/step - accuracy: 0.9991 - loss: 0.0046 - val_accuracy: 1.0000 - val_loss: 0.0049
Epoch 28/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 549ms/step - accuracy: 0.9997 - loss: 0.0038 - val_accuracy: 0.9950 - val_loss: 0.0088
Epoch 29/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 545ms/step - accuracy: 0.9994 - loss: 0.0039 - val_accuracy: 0.9950 - val_loss: 0.0082
Epoch 30/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 54s 540ms/step - accuracy: 0.9994 - loss: 0.0039 - val_accuracy: 0.9975 - val_loss: 0.0139
Epoch 31/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 545ms/step - accuracy: 0.9997 - loss: 0.0039 - val_accuracy: 1.0000 - val_loss: 0.0053
Epoch 32/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 55s 546ms/step - accuracy: 1.0000 - loss: 0.0020 - val_accuracy: 0.9975 - val_loss: 0.0060
```



In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

# Model Evaluation

In [ ]:
y_true = test_data.classes

y_pred_probs = model.predict(test_data)
y_pred = np.argmax(y_pred_probs, axis=1)

class_names = list(test_data.class_indices.keys())

In [ ]:
print(classification_report(y_true, y_pred, target_names=class_names))



```
                  precision    recall  f1-score   support

     Anthracnose       0.98      1.00      0.99        50
Bacterial Canker       1.00      1.00      1.00        50
  Cutting Weevil       1.00      1.00      1.00        50
        Die Back       1.00      0.98      0.99        50
      Gall Midge       1.00      1.00      1.00        50
         Healthy       1.00      1.00      1.00        50
  Powdery Mildew       1.00      1.00      1.00        50
     Sooty Mould       1.00      1.00      1.00        50

        accuracy                           1.00       400
       macro avg       1.00      1.00      1.00       400
    weighted avg       1.00      1.00      1.00       400

```



In [ ]:
cm = confusion_matrix(y_true, y_pred)
print(cm)



```
[[50  0  0  0  0  0  0  0]
 [ 0 50  0  0  0  0  0  0]
 [ 0  0 50  0  0  0  0  0]
 [ 1  0  0 49  0  0  0  0]
 [ 0  0  0  0 50  0  0  0]
 [ 0  0  0  0  0 50  0  0]
 [ 0  0  0  0  0  0 50  0]
 [ 0  0  0  0  0  0  0 50]]
```



In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
model.save('/content/mango_leaf_model.keras')
print("Model saved as mango_leaf_model.keras")

# PLOTING

In [ ]:
df = pd.read_csv("training_log.csv")

In [ ]:
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10
})

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(df['epoch'], df['accuracy'], marker='o', linewidth=2, label='Train Accuracy')
plt.plot(df['epoch'], df['val_accuracy'], marker='s', linewidth=2, linestyle='--', label='Validation Accuracy')

plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')


plt.grid(which='major', linestyle='-', linewidth=0.7)
plt.grid(which='minor', linestyle=':', linewidth=0.5)


plt.gca().xaxis.set_major_locator(MultipleLocator(2))
plt.gca().xaxis.set_minor_locator(MultipleLocator(1))
plt.gca().yaxis.set_major_locator(MultipleLocator(0.1))
plt.gca().yaxis.set_minor_locator(MultipleLocator(0.02))

plt.legend()
plt.tight_layout()


plt.savefig("accuracy_plot.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:

plt.figure(figsize=(10, 6))

plt.plot(df['epoch'], df['loss'], marker='o', linewidth=2, label='Train Loss')
plt.plot(df['epoch'], df['val_loss'], marker='s', linewidth=2, linestyle='--', label='Validation Loss')

plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')


plt.grid(which='major', linestyle='-', linewidth=0.7)
plt.grid(which='minor', linestyle=':', linewidth=0.5)


plt.gca().xaxis.set_major_locator(MultipleLocator(2))
plt.gca().xaxis.set_minor_locator(MultipleLocator(1))
plt.gca().yaxis.set_major_locator(MultipleLocator(0.5))
plt.gca().yaxis.set_minor_locator(MultipleLocator(0.1))

plt.legend()
plt.tight_layout()


plt.savefig("loss_plot.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


cm = np.array([
    [50, 0, 0, 0, 0, 0, 0, 0],
    [0, 50, 0, 0, 0, 0, 0, 0],
    [0, 0, 50, 0, 0, 0, 0, 0],
    [1, 0, 0, 49, 0, 0, 0, 0],
    [0, 0, 0, 0, 50, 0, 0, 0],
    [0, 0, 0, 0, 0, 50, 0, 0],
    [0, 0, 0, 0, 0, 0, 50, 0],
    [0, 0, 0, 0, 0, 0, 0, 50]
])


classes = [
    "Anthracnose", "Bacterial Canker", "Cutting Weevil", "Die Back",
    "Gall Midge", "Healthy", "Powdery Mildew", "Sooty Mould"
]


fig, ax = plt.subplots(figsize=(8, 7))


im = ax.imshow(cm, cmap="Blues")


ax.set_xticks(np.arange(len(classes)))
ax.set_yticks(np.arange(len(classes)))

ax.set_xticklabels(classes, rotation=30, ha="right")
ax.set_yticklabels(classes)

ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("Confusion Matrix")


threshold = cm.max() / 2

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f"{cm[i, j]}",
                ha="center", va="center",
                color="white" if cm[i, j] > threshold else "black",
                fontsize=10)


ax.set_xticks(np.arange(-.5, len(classes), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(classes), 1), minor=True)
ax.grid(which="minor", linestyle='-', linewidth=0.5)
ax.tick_params(which="minor", bottom=False, left=False)


cbar = fig.colorbar(im)
cbar.ax.set_ylabel("Number of Samples", rotation=90)

plt.tight_layout()


plt.savefig("confusion_matrix_blue.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


data = {
    "Class": [
        "Anthracnose", "Bacterial Canker", "Cutting Weevil", "Die Back",
        "Gall Midge", "Healthy", "Powdery Mildew", "Sooty Mould"
    ],
    "Precision": [0.98, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00],
    "Recall":    [1.00, 1.00, 1.00, 0.98, 1.00, 1.00, 1.00, 1.00],
    "F1-score":  [0.99, 1.00, 1.00, 0.99, 1.00, 1.00, 1.00, 1.00],
}

df = pd.DataFrame(data)
df.set_index("Class", inplace=True)


fig, ax = plt.subplots(figsize=(10, 6))


cmap = "coolwarm"

cax = ax.imshow(df.values, aspect='auto', cmap=cmap)


ax.set_xticks(np.arange(len(df.columns)))
ax.set_yticks(np.arange(len(df.index)))

ax.set_xticklabels(df.columns)
ax.set_yticklabels(df.index)

plt.setp(ax.get_xticklabels(), rotation=30, ha="right")


for i in range(len(df.index)):
    for j in range(len(df.columns)):
        ax.text(j, i, f"{df.iloc[i, j]:.2f}",
                ha="center", va="center",
                color="black", fontsize=10)


ax.set_title("Classification Report Heatmap")


fig.colorbar(cax)


ax.set_xticks(np.arange(-.5, len(df.columns), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(df.index), 1), minor=True)
ax.grid(which="minor", linestyle='-', linewidth=0.5)
ax.tick_params(which="minor", bottom=False, left=False)

plt.tight_layout()


plt.savefig("classification_report_heatmap.png", dpi=300, bbox_inches='tight')

plt.show()

# LIME

In [ ]:
model = load_model('/content/drive/MyDrive/mango_leaf_model.keras')

In [ ]:
IMG_SIZE = (224, 224)


class_names = [
    "Anthracnose", "Bacterial Canker", "Cutting Weevil", "Die Back",
    "Gall Midge", "Healthy", "Powdery Mildew", "Sooty Mould"
]


img_path = "/content/IMG_20211108_120232 (Custom).jpg"
img = image.load_img(img_path, target_size=IMG_SIZE)
img_array = image.img_to_array(img).astype(np.float32) / 255.0


img_for_lime = img_array.copy()


def predict_fn(images):
    images = np.array(images)
    return model.predict(images)


pred_probs = model.predict(np.expand_dims(img_array, axis=0))
pred_class = np.argmax(pred_probs)
confidence = np.max(pred_probs)
pred_label = class_names[pred_class]


explainer = lime_image.LimeImageExplainer()

explanation = explainer.explain_instance(
    img_for_lime,
    predict_fn,
    top_labels=1,
    hide_color=0,
    num_samples=1000
)


temp, mask = explanation.get_image_and_mask(
    label=pred_class,
    positive_only=True,
    num_features=10,
    hide_rest=False
)


plt.figure(figsize=(8, 6), dpi=300)

plt.imshow(mark_boundaries(temp, mask))
plt.axis('off')


plt.suptitle(f"LIME Explanation: {pred_label} ({confidence*100:.2f}%)",
             fontsize=14)

plt.tight_layout()


plt.savefig("lime_explanation.png", dpi=300, bbox_inches='tight')

plt.show()